# Deduplicate Master_List.csv by shared email address

Two rows can represent the same FO under different organisation names (e.g. `Verny Capital` vs `Verny Capital / Bulat Utemuratov`). We detect this by matching genuine email addresses in the `Email address` column (ignoring placeholder text like "no email captured" or "contact form / phone route").

Rows can be linked through a chain of different shared emails (A↔B via email 1, B↔C via email 2), so duplicates are grouped using connected components, not simple pairwise matching.

Strategy: for each duplicate group, keep the row sourced from the original Master Longlist as primary, fill any blank fields from the other row(s), record other organisation names in a new `Alternate Names` column, and merge `Source File` provenance. Result overwrites `data/Master_List.csv` (a backup is taken first).

In [1]:
import re
import shutil
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data") / "Master_List.csv"
MASTER_SOURCE_FILE = "TBP_Family_Office_Generated_Lists_Fresh_Master.xlsx"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(158, 31)


,Region,Country,Organisation,Prospect Category,HQ / Primary Geography,Address / Office Location,Family / Founder / Strategic Nature,Known Sector Themes,TBP / Regional Corridor Relevance,Possible TBP Entry Point,...,Pipeline Stage,Scoring Status,Public Source URLs,Notes / Diligence Flags,Email address,Contact Email Status,Contact Email Source URLs,Contact Email Notes,Contact Enrichment Date,Source File
0,Central Asia,Kazakhstan,Verny Capital / Bulat Utemuratov,Private investment group / family-capital styl...,Almaty / Astana,NaN,Private investment group / family-capital styl...,Family office services / private wealth / stew...,Leader in private investments in Kazakhstan; i...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://vernycapital.com/en/bulat-utemuratov-i...,Apply enhanced due diligence where political e...,info@vernycapital.com,Official/public email - verify before outreach,https://vernycapital.com/en/bulat-utemuratov-i...,Use general office email or warm introduction;...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
1,Central Asia,Kazakhstan,BI Group / Aidyn Rakhimbayev,"Investment, development, construction and educ...",Astana,NaN,"Investment, development, construction and educ...",Family office services / private wealth / stew...,"City development, real estate, construction an...",Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://bi.group/en/company,Apply enhanced due diligence where political e...,ipm@bi.group,Official/public email - verify before outreach,https://bi.group/en/company,Use institutional or partnership route; verify...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
2,Central Asia,Kazakhstan,Lancaster Group,Diversified international holding company,Almaty,NaN,Diversified international holding company,"Mining, infrastructure, energy, financial serv...","Industrial infrastructure, oil and gas service...",Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://lancasterholding.com/en,Apply enhanced due diligence where political e...,info@lgk.kz,Official/public email - verify before outreach,https://lancasterholding.com/en,Use general office email; verify current conta...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
3,Central Asia,Kazakhstan,Resmi Group,Diversified investment holding,Kazakhstan / Central Asia,NaN,Diversified investment holding,Family office services / private wealth / stew...,Diversified investment holding operating in Ka...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://kz.linkedin.com/company/group-of-compa...,Apply enhanced due diligence where political e...,inform@resmi.kz,Official/public email - verify before outreach,https://kase.kz/en/listing/issuers/RESC/,Issuer/contact source; verify best route for c...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
4,Central Asia,Kazakhstan,AIFC Family Office Framework,Family office jurisdiction / structuring platform,Astana,NaN,Family office jurisdiction / structuring platform,Family office services / private wealth / stew...,Institutional route for structuring Central As...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://afsa.aifc.kz/aifc-introduces-family-of...,Apply enhanced due diligence where political e...,info@afsa.kz; consultation@afsa.kz; registrati...,Official public institutional contacts,https://afsa.aifc.kz/aifc-introduces-family-of...,"Use as jurisdiction/regulatory gateway, not a ...",2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...


## Extract genuine emails per row

`Email address` can hold multiple addresses separated by `;`, or placeholder text ("no email captured", "contact form / phone route", etc). Only keep tokens that actually look like an email address.

In [2]:
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


def extract_emails(value):
    if pd.isna(value):
        return []
    tokens = [t.strip().lower() for t in str(value).split(";") if t.strip()]
    return [t for t in tokens if EMAIL_RE.match(t)]


df["_emails"] = df["Email address"].apply(extract_emails)

n_with_email = (df["_emails"].str.len() > 0).sum()
print(f"{n_with_email} / {len(df)} rows have at least one genuine email")

112 / 158 rows have at least one genuine email


## Group rows into duplicate clusters (connected components)

In [3]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


uf = UnionFind(len(df))

email_to_first_row = {}
for pos, emails in enumerate(df["_emails"]):
    for e in emails:
        if e in email_to_first_row:
            uf.union(pos, email_to_first_row[e])
        else:
            email_to_first_row[e] = pos

clusters = {}
for pos in range(len(df)):
    root = uf.find(pos)
    clusters.setdefault(root, []).append(pos)

duplicate_clusters = [idxs for idxs in clusters.values() if len(idxs) > 1]
print(f"{len(duplicate_clusters)} duplicate clusters found, covering {sum(len(c) for c in duplicate_clusters)} rows")

for idxs in duplicate_clusters:
    print(idxs, "->", df.iloc[idxs]["Organisation"].tolist())

8 duplicate clusters found, covering 17 rows
[0, 143] -> ['Verny Capital / Bulat Utemuratov', 'Verny Capital']
[1, 141] -> ['BI Group / Aidyn Rakhimbayev', 'BI Group']
[2, 146] -> ['Lancaster Group', 'Lancaster Group']
[6, 148] -> ['Ordabasy Group', 'Ordabasy Group']
[12, 145] -> ['Kusto Group / Yerkin Tatishev', 'Kusto Group']
[14, 127, 128] -> ['Artel Electronics / AKFA Group / Jahongir Artikkhodjaev-linked ecosystem', 'AKFA Group', 'Artel Electronics']
[15, 132] -> ['Murad Buildings / Murad Nazarov', 'Murad Buildings']
[25, 33] -> ['Widjaja Family Investment Office / Sinar Mas Group / SMDV', 'SMDV — Sinar Mas Digital Ventures']


## Merge each cluster into a single row

Primary = the row sourced from the original Master Longlist (falls back to the lowest-index row if no master row is in the cluster). Blank fields on the primary get filled from the other rows in the cluster; other organisation names are recorded in `Alternate Names`; `Source File` becomes the union of all contributing sources.

In [4]:
def is_blank(val):
    return pd.isna(val) or (isinstance(val, str) and not val.strip())


alt_names_col = pd.Series(pd.NA, index=df.index, dtype="object")
rows_to_drop = []

for idxs in duplicate_clusters:
    rows = df.loc[idxs]
    master_rows = rows[rows["Source File"] == MASTER_SOURCE_FILE]
    primary = master_rows.index[0] if len(master_rows) else min(idxs)
    others = [i for i in idxs if i != primary]

    for col in df.columns:
        if col == "_emails":
            continue
        if is_blank(df.loc[primary, col]):
            for o in others:
                other_val = df.loc[o, col]
                if not is_blank(other_val):
                    df.loc[primary, col] = other_val
                    break

    alt_names = [
        df.loc[o, "Organisation"] for o in others
        if df.loc[o, "Organisation"] != df.loc[primary, "Organisation"]
    ]
    if alt_names:
        alt_names_col.loc[primary] = "; ".join(dict.fromkeys(alt_names))

    df.loc[primary, "Source File"] = "; ".join(dict.fromkeys(rows["Source File"].tolist()))
    rows_to_drop.extend(others)

df["Alternate Names"] = alt_names_col

deduped_df = df.drop(index=rows_to_drop).drop(columns=["_emails"]).reset_index(drop=True)

print("Before:", len(df))
print("Removed:", len(rows_to_drop))
print("After:", len(deduped_df))
deduped_df[deduped_df["Alternate Names"].notna()][["Organisation", "Alternate Names", "Source File", "Email address"]]

Before: 158
Removed: 9
After: 149


,Organisation,Alternate Names,Source File,Email address
0,Verny Capital / Bulat Utemuratov,Verny Capital,TBP_Family_Office_Generated_Lists_Fresh_Master...,info@vernycapital.com
1,BI Group / Aidyn Rakhimbayev,BI Group,TBP_Family_Office_Generated_Lists_Fresh_Master...,ipm@bi.group
12,Kusto Group / Yerkin Tatishev,Kusto Group,TBP_Family_Office_Generated_Lists_Fresh_Master...,info@kustogroup.com
14,Artel Electronics / AKFA Group / Jahongir Arti...,AKFA Group; Artel Electronics,TBP_Family_Office_Generated_Lists_Fresh_Master...,info@akfaaluminium.com; info@artelelectronics.com
15,Murad Buildings / Murad Nazarov,Murad Buildings,TBP_Family_Office_Generated_Lists_Fresh_Master...,pr@mbc.uz
25,Widjaja Family Investment Office / Sinar Mas G...,SMDV — Sinar Mas Digital Ventures,TBP_Family_Office_Generated_Lists_Fresh_Master...,care@sinarmas.org


## Back up and overwrite Master_List.csv

In [5]:
backup_path = DATA_PATH.with_name(DATA_PATH.stem + ".backup.csv")
if not backup_path.exists():
    shutil.copy(DATA_PATH, backup_path)
    print("Backed up ->", backup_path.resolve())
else:
    print("Backup already exists, skipping ->", backup_path.resolve())

deduped_df.to_csv(DATA_PATH, index=False)
print("Wrote", DATA_PATH.resolve(), "rows:", len(deduped_df))

Backed up -> C:\Users\USER\Desktop\TBP\tbp-dashboard\data\Master_List.backup.csv
Wrote C:\Users\USER\Desktop\TBP\tbp-dashboard\data\Master_List.csv rows: 149
